In [4]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\HTOC\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\HTOC\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [ ]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import urllib.parse

SPECIFIC_INDICATORS = [
    "80.94.92.171",    "187.141.210.92",  "64.62.197.39",    "65.49.1.114",
    "65.49.1.44",      "64.62.197.52",    "65.49.1.199",     "142.93.115.5",
    "65.49.1.84",      "65.49.1.179",     "65.49.1.167",     "65.49.1.148",
    "8.34.210.60",     "167.99.13.19",    "65.49.1.49",      "65.49.1.63",
    "65.49.1.121",     "65.49.1.83",      "64.62.156.168",   "65.49.1.19",
    "65.49.1.37",      "65.49.1.217",     "193.163.125.151", "64.62.156.183",
    "64.62.156.196",   "65.49.1.109",     "65.49.1.166",     "64.62.156.22",
    "64.62.156.33",    "192.227.183.134", "198.235.24.87",   "64.62.156.98",
    "198.235.24.193",  "198.235.24.108",  "91.224.92.99",    "64.62.156.162",
    "65.49.1.102",     "64.62.197.41",    "64.62.156.230",   "65.49.1.129",
    "65.49.1.135",     "65.49.1.233",     "65.49.1.105",     "65.49.1.85",
    "64.62.156.106",   "65.49.1.193",     "65.49.1.175",     "65.49.1.116",
    "65.49.1.103",     "64.62.197.101",   "65.49.1.230",     "65.49.1.238",
    "65.49.1.211",     "65.49.1.117",     "65.49.1.81",      "65.49.1.53",
    "65.49.1.236",     "65.49.1.130",     "65.49.1.46",      "64.62.156.193",
    "65.49.1.99",      "64.62.197.38",    "172.210.68.2",    "64.62.197.226",
    "65.49.1.112",     "65.49.1.231",     "65.49.1.194",     "65.49.1.123",
    "65.49.1.222",     "64.62.156.61",    "185.247.137.163", "64.23.214.73",
    "64.62.197.71",    "65.49.1.25",      "65.49.1.169",     "65.49.1.218",
    "65.49.1.157",     "64.62.156.127",   "65.49.1.159",     "65.49.1.182",
    "64.62.197.151",   "167.94.146.51",   "65.49.1.97",      "87.236.176.163",
    "65.49.1.98",      "64.62.156.108",   "65.49.1.89",      "64.62.197.80",
    "207.90.244.3",    "137.184.112.192", "65.49.1.172",     "40.124.175.75",
    "20.221.72.102",   "65.49.1.146",     "87.236.176.167",  "62.60.130.210",
    "204.76.203.201",  "116.124.133.221", "83.254.239.226",  "64.225.74.178",
    "199.45.154.159",  "65.49.1.43",      "87.236.176.164",  "65.49.1.161",
    "64.62.156.79",    "134.209.126.135", "109.70.100.1",    "65.49.1.28",
    "14.241.100.20",   "172.105.147.201", "65.49.1.234",     "185.247.137.172",
    "193.37.69.113",   "78.39.48.166",    "210.56.48.242",   "167.94.138.167",
    "103.99.3.226",    "65.49.1.229",     "87.236.176.158",  "205.210.31.106",
    "45.144.212.221",  "65.49.1.202",     "125.209.101.162", "185.47.14.233",
    "207.90.244.5",    "15.156.234.60",
]
RESULT_PAGE_SIZE = 10000
MAX_WORKERS = 3  # parallel page fetches; raise carefully to avoid rate limits

indicator_condition = ", ".join([f'"{i}"' for i in SPECIFIC_INDICATORS])
tql_raw = f'summary IN ({indicator_condition})'
tql_encoded = urllib.parse.quote(tql_raw)

def fetch_page(offset):
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_request_uri(
        f'/v3/indicators?tql={tql_encoded}'
        f'&fields=tags,observations,attributes,associatedGroups'
        f'&resultStart={offset}&resultLimit={RESULT_PAGE_SIZE}'
    )
    response = tc.api_request(ro)
    ct = response.headers.get('content-type', '')
    if not ct.startswith('application/json'):
        raise RuntimeError(f"Non-JSON response ({ct}): {response.content[:200]}")
    return response.json()

# First page — establishes total count and seeds data
try:
    first = fetch_page(0)
except Exception as e:
    display(f"Failed on first page: {e}")
    raise

total_count = first.get('count', 0)
display(f"Total matching indicators: {total_count}")

normalized_data = [
    item for item in first.get('data', [])
    if isinstance(item, dict) and 'summary' in item
]

# Fetch remaining pages in parallel
remaining_offsets = range(RESULT_PAGE_SIZE, total_count, RESULT_PAGE_SIZE)

if remaining_offsets:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(fetch_page, o): o for o in remaining_offsets}
        for future in as_completed(futures):
            offset = futures[future]
            try:
                page = future.result()
                normalized_data.extend([
                    item for item in page.get('data', [])
                    if isinstance(item, dict) and 'summary' in item
                ])
            except Exception as e:
                display(f"Failed to fetch page at offset {offset}: {e}")

display(f"Collected {len(normalized_data)} indicators")

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].str.split(n=1).str[0].str.strip()
    observed_src = observed_src.drop_duplicates(subset='indicator').reset_index(drop=True)

display(observed_src)


In [ ]:
# Unnest attributes
attr_col = 'attributes.data'
if attr_col in observed_src.columns:
    attr_exploded = (
        observed_src
        .explode(attr_col)
        .reset_index(drop=True)
    )
    attr_flat = pd.json_normalize(
        attr_exploded[attr_col].apply(lambda x: x if isinstance(x, dict) else {})
    )
    result = pd.concat([
        attr_exploded.drop(columns=[attr_col]),
        attr_flat.add_prefix('attr.')
    ], axis=1)
else:
    result = observed_src.copy()

result = result.drop_duplicates(subset='indicator').reset_index(drop=True)

# Drop unwanted columns
drop_cols = [
    'attr.type', 'attr.pinned', 'attr.default',
    'attr.createdBy.id', 'attr.id',
    'activeLocked', 'active', 'privateFlag',
]
result = result.drop(columns=[c for c in drop_cols if c in result.columns])

# Clean up column names
rename_map = {
    'id':                          'tc_id',
    'dateAdded':                   'date_added',
    'ownerId':                     'owner_id',
    'ownerName':                   'owner',
    'webLink':                     'web_link',
    'type':                        'indicator_type',
    'lastModified':                'last_modified',
    'lastObserved':                'last_observed',
    'legacyLink':                  'legacy_link',
    'tags.data':                   'tags',
    'ip':                          'ip',
    'attr.value':                  'attribute',
    'attr.dateAdded':              'attribute_date_added',
    'attr.lastModified':           'attribute_last_modified',
    'attr.source':                 'attribute_source',
    'attr.createdBy.userName':     'attribute_created_by',
    'attr.createdBy.firstName':    'attribute_created_by_first',
    'attr.createdBy.lastName':     'attribute_created_by_last',
    'attr.createdBy.pseudonym':    'attribute_created_by_pseudonym',
    'attr.securityLabels.data':    'attribute_security_labels',
}
result = result.rename(columns={k: v for k, v in rename_map.items() if k in result.columns})

display(result)